In [25]:
import numpy as np
from typing import List, Tuple, Union, Optional

In [42]:
class TensorTrainDecomposition(object):

    def __init__(self, data: Union[List[np.ndarray], np.ndarray], rank = None):

        if isinstance(data, list):
            if np.all([data[i].ndim == 3 for i in range(len(data))]):
                self.ndim = len(data)
                self.shape = tuple(data[i].shape[1] for i in range(len(data)))
                self.rank = tuple([data[i].shape[0] for i in range(len(data))]+[data[-1].shape[-1]])
                self.cores = data
            else:
                raise ValueError('All cores must be third-order.')

        elif isinstance(data, np.ndarray):
              self.ndim = data.ndim
              self.shape = data.shape
              if rank is None:
                  self.rank = [1] * (self.ndim + 1)
                  for i in range(self.ndim):
                     mat = self._unflold(data, i)
                     s = np.linalg.svd(mat, compute_uv=False)
                     self.rank[i+1] = np.sum(s > np.finfo(float).eps) # tolerance can be changed here
                  self.rank = tuple(self.rank)
              else:
                  self.rank = tuple(rank) # other checks can be done later

              self.cores = self._decompose(data)
        else:
            raise TypeError('data must be either as a list of third-order TT cores or an ndarray.')

    def __repr__(self):
        return ('\n'
                'Tensor Train: order = {ndim}, \n'
                '              shape = {shape}, \n'
                '              rank = {rank}'.format(ndim=self.ndim, shape=self.shape, rank=self.rank))

    def full(self):
        cores = self.cores
        rank = self.rank
        shape = self.shape
        tensor = cores[0]
        for k in range(1, len(cores)):
            tensor = np.reshape(tensor, (tensor.size // rank[k], rank[k]))
            core_k = cores[k]
            core_k = np.reshape(core_k, (rank[k], shape[k] * rank[k + 1]))
            tensor = np.dot(tensor, core_k)
        tensor = np.reshape(tensor, shape)
        return tensor

    # Utility functions
    def _truncate_svd(self, matrix, rank):
        U, S, Vh = np.linalg.svd(matrix, full_matrices=False)
        U_trunc = U[:, :rank]
        S_trunc = S[:rank]
        Vh_trunc = Vh[:rank, :]
        return U_trunc, np.diag(S_trunc), Vh_trunc

    def _unflold(self, tensor, mode):
        shape = tensor.shape
        if mode == 0:
            return tensor.reshape((shape[0], np.prod(shape[1:])))
        elif 0 < mode < len(shape)-1:
            return tensor.reshape((np.prod(shape[:mode+1]), np.prod(shape[mode+1:])))
        elif mode == len(shape)-1:
            return tensor.reshape(-1,1)
        else:
            raise ValueError('mode must lie between 0 and {}'.format(len(shape)-1))

    def _flold(self, tensor, shape):
        return tensor.reshape(shape)

    def _decompose(self, data):
        """Perform tensor train decomposition."""
        cores = []
        tensor = data.copy()
        rank = self.rank
        matrix = self._unflold(tensor, 0)
        shape = self.shape
        for k in range(self.ndim - 1):
            U, S, Vh = self._truncate_svd(matrix, rank[k + 1])
            core_shape = (rank[k], shape[k], rank[k + 1])
            core_k = U.reshape(core_shape)
            cores.append(core_k)
            matrix = np.dot(S, Vh).reshape(rank[k + 1]*shape[k+1], -1)
        cores.append(matrix.reshape((rank[-2], shape[-1], rank[-1])))

        return cores

In [59]:
tensor = np.random.rand(4, 5, 6, 7)
TTrank = [1, 2, 3, 4, 1]  # Desired TT rank

In [60]:
TTD = TensorTrainDecomposition(tensor, TTrank)

In [61]:
TTD


Tensor Train: order = 4, 
              shape = (4, 5, 6, 7), 
              rank = (1, 2, 3, 4, 1)

In [62]:
TTD.cores[0].shape

(1, 4, 2)

In [63]:
y = TensorTrainDecomposition(TTD.full())

In [64]:
y


Tensor Train: order = 4, 
              shape = (4, 5, 6, 7), 
              rank = (1, 4, 15, 7, 1)

In [65]:
np.linalg.norm(TTD.full() - y.full())

3.215917049268056e-14